In [1]:
import numpy as np
import pandas as pd

dt1_time = pd.read_csv("./datasets_new/DT1_log-timeline.csv")
dt1_final = pd.read_csv("./datasets_new/DT1_log-final.csv")

In [10]:
print(len(dt1_final["car_id"].unique()))
print(len(dt1_time["car_id"].unique()))

528
17084


In [ ]:
def simple_eda(cl_time_path, cl_final_path):
    time_df = pd.read_csv(cl_time_path)
    final_df = pd.read_csv(cl_final_path)

    print("car_id ")

In [11]:
dt1_time["car_id"].unique

<bound method Series.unique of 0          606832
1          602829
2          612326
3          606925
4          613013
            ...  
1790784     17552
1790785     13951
1790786     17224
1790787     17743
1790788     17405
Name: car_id, Length: 1790789, dtype: int64>

In [4]:
dt1_time.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1790789 entries, 0 to 1790788
Data columns (total 14 columns):
 #   Column           Dtype  
---  ------           -----  
 0   timestamp        float64
 1   kst_date         object 
 2   kst_time         object 
 3   weather_rain     float64
 4   weather_fog      float64
 5   car_id           int64  
 6   car_type         object 
 7   lat              float64
 8   lon              float64
 9   vel              float64
 10  heading          float64
 11  park_slot_id     object 
 12  park_enter_time  float64
 13  park_exit_time   float64
dtypes: float64(9), int64(1), object(4)
memory usage: 191.3+ MB


In [15]:
import pandas as pd
import os
from glob import glob

# =========================
# 1. 데이터 로드
# =========================

base_path = "./datasets_new"

timeline_files = sorted(glob(os.path.join(base_path, "*timeline.csv")))
final_files = sorted(glob(os.path.join(base_path, "*final.csv")))

datasets = {}

for t_file, f_file in zip(timeline_files, final_files):

    client_id = os.path.basename(t_file).split("_")[0]   # DT1, DT2 ...

    timeline = pd.read_csv(t_file)
    final = pd.read_csv(f_file)

    timeline["client_id"] = client_id
    final["client_id"] = client_id

    datasets[client_id] = {
        "timeline": timeline,
        "final": final
    }

print("로드 완료:", datasets.keys())


# =========================
# 2. 전체 EDA 함수
# =========================

def run_basic_eda(client_id, timeline, final):

    print("\n" + "="*70)
    print(f"[ {client_id} ]")
    print("="*70)

    # -------------------------------------------------
    # 1. car_id 중복/개수 확인
    # -------------------------------------------------

    print("\n[1] car_id 통계")

    timeline_unique = timeline["car_id"].nunique()

    # final 컬럼 자동 탐색
    final_car_col = None

    for col in final.columns:
        if "car" in col.lower():
            final_car_col = col
            break

    if final_car_col is not None:
        final_unique = final[final_car_col].astype(str).nunique()
    else:
        final_unique = "car_id 컬럼 없음"

    print(f"timeline unique car_id : {timeline_unique}")
    print(f"final unique car_id    : {final_unique}")


    # -------------------------------------------------
    # 2. final car_id가 timeline에 존재하는지
    # -------------------------------------------------

    print("\n[2] final car_id 존재 여부")

    if final_car_col is not None:

        # car_id_slot 형태 처리
        if "slot" in final_car_col.lower():

            final["parsed_car_id"] = (
                final[final_car_col]
                .astype(str)
                .str.extract(r'(\d+)')[0]
            )

            final_ids = set(final["parsed_car_id"].dropna().astype(int))

        else:
            final_ids = set(final[final_car_col].dropna().astype(int))

        timeline_ids = set(timeline["car_id"].dropna().astype(int))

        matched = len(final_ids & timeline_ids)
        missing = len(final_ids - timeline_ids)
        extra = len(timeline_ids - final_ids)

        print(f"주차 안한 차량 수 : {extra}")

        print(f"timeline에 존재하는 final car_id : {matched}")
        print(f"timeline에 없는 final car_id    : {missing}")

    else:
        print("car_id 관련 컬럼 없음")


    # -------------------------------------------------
    # 3. slot_id 분포
    # -------------------------------------------------

    print("\n[3] slot_id 분포")

    slot_cols = [c for c in timeline.columns if "slot" in c.lower()]

    if len(slot_cols) > 0:

        slot_col = slot_cols[0]

        slot_counts = (
            timeline[slot_col]
            .value_counts(dropna=False)
            .head(10)
        )

        print(slot_counts)

    else:
        print("slot 컬럼 없음")


    # -------------------------------------------------
    # 4. timestamp 범위
    # -------------------------------------------------

    print("\n[4] timestamp 범위")

    timeline["datetime"] = pd.to_datetime(
        timeline["kst_date"].astype(str) + " " +
        timeline["kst_time"].astype(str)
        )
     
    print("min :", timeline["datetime"].min())
    print("max :", timeline["datetime"].max())



    # -------------------------------------------------
    # 5. 차량별 row 수 분포
    # -------------------------------------------------

    print("\n[5] 차량별 timeline row 수 분포")

    row_counts = timeline.groupby("car_id").size()

    print(row_counts.describe())

    print("\n상위 10개 차량")
    print(row_counts.sort_values(ascending=False).head(10))


# =========================
# 3. 전체 DT 실행
# =========================

for client_id, data in datasets.items():

    run_basic_eda(
        client_id,
        data["timeline"],
        data["final"]
    )

로드 완료: dict_keys(['DT1', 'DT2', 'DT3'])

[ DT1 ]

[1] car_id 통계
timeline unique car_id : 17084
final unique car_id    : 528

[2] final car_id 존재 여부
주차 안한 차량 수 : 16556
timeline에 존재하는 final car_id : 528
timeline에 없는 final car_id    : 0

[3] slot_id 분포
park_slot_id
NaN     740134
A000     52994
A001     52883
A002     52861
A004     52817
A013     52812
A009     52779
A018     52741
A005     52710
A008     52676
Name: count, dtype: int64

[4] timestamp 범위
min : 2026-03-24 08:50:56
max : 2026-03-24 23:59:59

[5] 차량별 timeline row 수 분포
count    17084.000000
mean       104.822583
std        409.738095
min          2.000000
25%         38.000000
50%         44.000000
75%         47.000000
max       3743.000000
dtype: float64

상위 10개 차량
car_id
678      3743
181      3740
2664     3718
10295    3714
14693    3713
12771    3712
3016     3712
4932     3712
11394    3711
6038     3711
dtype: int64

[ DT2 ]

[1] car_id 통계
timeline unique car_id : 27027
final unique car_id    : 826

[2] final car_id 

In [ ]:
import pandas as pd
import os
from glob import glob

# =========================
# 1. 데이터 로드
# =========================

base_path = "./datasets_new"

timeline_files = sorted(glob(os.path.join(base_path, "*timeline.csv")))
final_files = sorted(glob(os.path.join(base_path, "*final.csv")))

datasets = {}


for t_file, f_file in zip(timeline_files, final_files):

    client_id = os.path.basename(t_file).split("_")[0]

    timeline = pd.read_csv(t_file)
    final = pd.read_csv(f_file)

    timeline["client_id"] = client_id
    final["client_id"] = client_id

    # datetime 생성
    timeline["datetime"] = pd.to_datetime(
        timeline["kst_date"].astype(str) + " " +
        timeline["kst_time"].astype(str)
    )

    datasets[client_id] = {
        "timeline": timeline,
        "final": final
    }

print("로드 완료:", datasets.keys())

In [ ]:



# =========================
# 2. 전체 EDA
# =========================

for client_id, data in datasets.items():

    timeline = data["timeline"]
    final = data["final"]

    print("\n" + "="*80)
    print(f"[ {client_id} ]")
    print("="*80)

    # =====================================================
    # 1. 기본 통계
    # =====================================================

    print("\n[1] 기본 통계")

    timeline_ids = set(timeline["car_id"].unique())

    final_car_col = [c for c in final.columns if "car" in c.lower()][0]

    if "slot" in final_car_col.lower():

        final["parsed_car_id"] = (
            final[final_car_col]
            .astype(str)
            .str.extract(r'(\d+)')[0]
            .astype(int)
        )

        final_ids = set(final["parsed_car_id"].unique())

    else:
        final_ids = set(final[final_car_col].unique())

    print(f"timeline 차량 수        : {len(timeline_ids):,}")
    print(f"실제 주차 차량 수      : {len(final_ids):,}")
    print(f"주차 안한 차량 수      : {len(timeline_ids-final_ids):,}")

    print(f"\n시간 범위")
    print(f"min : {timeline['datetime'].min()}")
    print(f"max : {timeline['datetime'].max()}")



    # =====================================================
    # 2. 슬롯 개수 확인
    # =====================================================

    print("\n[2] 슬롯 정보")

    slots = sorted(
        timeline["park_slot_id"]
        .dropna()
        .unique()
    )

    print(f"총 슬롯 개수 : {len(slots)}")

    print("\n슬롯 목록")
    print(slots)



    # =====================================================
    # 3. 슬롯 점유 통계
    # =====================================================

    print("\n[3] 슬롯 점유 통계")

    slot_counts = (
        timeline["park_slot_id"]
        .value_counts(dropna=False)
    )

    print(slot_counts.head(15))



    # =====================================================
    # 4. timestamp 간격 확인
    # =====================================================

    print("\n[4] timestamp 간격")

    time_diff = (
        timeline["timestamp"]
        .sort_values()
        .diff()
        .dropna()
    )

    print(time_diff.value_counts().head())



    # =====================================================
    # 5. 차량별 trajectory 길이(이동 경로)
    # =====================================================

    print("\n[5] 차량별 trajectory 길이")

    row_counts = timeline.groupby("car_id").size()

    print(row_counts.describe())

    print("\n상위 10개")
    print(row_counts.sort_values(ascending=False).head(10))



    # =====================================================
    # 6. 속도 분포
    # =====================================================

    print("\n[6] 속도 분포")

    print(timeline["vel"].describe())



    # =====================================================
    # 7. 시간대별 차량 수
    # =====================================================

    print("\n[7] 시간대별 차량 수")

    traffic = (
        timeline
        .groupby(timeline["datetime"].dt.hour)["car_id"]
        .nunique()
    )

    print(traffic)



    # =====================================================
    # 8. duration 분포
    # =====================================================

    print("\n[8] 체류시간(duration)")

    duration_col = [
        c for c in final.columns
        if "duration" in c.lower()
    ]

    if len(duration_col) > 0:

        dcol = duration_col[0]

        print(final[dcol].describe())

    else:
        print("duration 컬럼 없음")


    print("\n")

로드 완료: dict_keys(['DT1', 'DT2', 'DT3'])

[ DT1 ]

[1] 기본 통계
timeline 차량 수        : 17,084
실제 주차 차량 수      : 528
주차 안한 차량 수      : 16,556

시간 범위
min : 2026-03-24 08:50:56
max : 2026-03-24 23:59:59

[2] 슬롯 정보
총 슬롯 개수 : 20

슬롯 목록
['A000', 'A001', 'A002', 'A003', 'A004', 'A005', 'A006', 'A007', 'A008', 'A009', 'A010', 'A011', 'A012', 'A013', 'A014', 'A015', 'A016', 'A017', 'A018', 'A019']

[3] 슬롯 점유 통계
park_slot_id
NaN     740134
A000     52994
A001     52883
A002     52861
A004     52817
A013     52812
A009     52779
A018     52741
A005     52710
A008     52676
A007     52654
A010     52599
A015     52509
A012     52499
A017     52396
Name: count, dtype: int64

[4] timestamp 간격
timestamp
0.000000    1736253
1.000032         43
0.999982         38
0.999977         37
1.000008         36
Name: count, dtype: int64

[5] 차량별 trajectory 길이
count    17084.000000
mean       104.822583
std        409.738095
min          2.000000
25%         38.000000
50%         44.000000
75%         47.000000
max

In [9]:
import pandas as pd
import os
from glob import glob

# =========================
# 1. 데이터 로드
# =========================

base_path = "../data/raw"

timeline_files = sorted(glob(os.path.join(base_path, "*timeline.csv")))
final_files = sorted(glob(os.path.join(base_path, "*final.csv")))

datasets = {}


for t_file, f_file in zip(timeline_files, final_files):

    client_id = os.path.basename(t_file).split("_")[0]

    timeline = pd.read_csv(t_file)
    final = pd.read_csv(f_file)

    timeline["client_id"] = client_id
    final["client_id"] = client_id

    # datetime 생성
    timeline["datetime"] = pd.to_datetime(
        timeline["kst_date"].astype(str) + " " +
        timeline["kst_time"].astype(str)
    )

    datasets[client_id] = {
        "timeline": timeline,
        "final": final
    }

print("로드 완료:", datasets.keys())

로드 완료: dict_keys(['DT1', 'DT2', 'DT3'])


In [ ]:
import pandas as pd

summary = []

for client_id, data in datasets.items():

    timeline = data["timeline"]
    final = data["final"]

    # datetime 생성 안되어 있으면
    if "datetime" not in timeline.columns:
        timeline["datetime"] = pd.to_datetime(
            timeline["kst_date"].astype(str)
            + " "
            + timeline["kst_time"].astype(str)
        )

    # -------------------------
    # 차량 수
    # -------------------------

    total_cars = timeline["car_id"].nunique()

    final_car_col = [
        c for c in final.columns
        if "car" in c.lower()
    ][0]

    if "slot" in final_car_col.lower():

        parking_cars = (
            final[final_car_col]
            .astype(str)
            .str.extract(r"(\d+)")[0]
            .nunique()
        )

    else:

        parking_cars = final[final_car_col].nunique()

    non_parking_cars = (
        total_cars - parking_cars
    )

    # -------------------------
    # 슬롯 수
    # -------------------------

    slot_count = (
        timeline["park_slot_id"]
        .dropna()
        .nunique()
    )

    # -------------------------
    # 시간 정보
    # -------------------------

    start_time = timeline["datetime"].min()
    end_time = timeline["datetime"].max()

    duration_hours = round(
        (end_time - start_time).total_seconds()
        / 3600,
        2
    )

    # -------------------------
    # 로그 수
    # -------------------------

    total_rows = len(timeline)

    summary.append({

        "Client": client_id,

        "Total Cars": total_cars,

        "Parking Cars": parking_cars,

        "Non-Parking Cars": non_parking_cars,

        "Slots": slot_count,

        "Timeline Rows": total_rows,

        "Start": start_time,

        "End": end_time,

        "Duration(H)": duration_hours
    })

summary_df = pd.DataFrame(summary)

print(summary_df)

  Client  Total Cars  Parking Cars  Non-Parking Cars  Slots  Timeline Rows  \
0    DT1       17084           528             16556     20        1790789   
1    DT2       27027           826             26201     20        2847371   
2    DT3       53407          1683             51724     20        5703852   

                Start                 End  Duration(H)  
0 2026-03-24 08:50:56 2026-03-24 23:59:59        15.15  
1 2026-03-25 00:00:00 2026-03-25 23:59:59        24.00  
2 2026-03-26 00:00:00 2026-03-27 23:59:59        48.00  


In [12]:
summary_df

,Client,Total Cars,Parking Cars,Non-Parking Cars,Slots,Timeline Rows,Start,End,Duration(H)
0,DT1,17084,528,16556,20,1790789,2026-03-24 08:50:56,2026-03-24 23:59:59,15.15
1,DT2,27027,826,26201,20,2847371,2026-03-25 00:00:00,2026-03-25 23:59:59,24.00
2,DT3,53407,1683,51724,20,5703852,2026-03-26 00:00:00,2026-03-27 23:59:59,48.00


In [15]:
slot_summary = []

for client_id, data in datasets.items():

    slots = sorted(
        data["timeline"]["park_slot_id"]
        .dropna()
        .unique()
    )

    slot_summary.append({
        "Client": client_id,
        "Slot Count": len(slots),
        "Slots": ", ".join(slots)
    })

slot_df = pd.DataFrame(slot_summary)

print(slot_df)

  Client  Slot Count                                              Slots
0    DT1          20  A000, A001, A002, A003, A004, A005, A006, A007...
1    DT2          20  A000, A001, A002, A003, A004, A005, A006, A007...
2    DT3          20  A000, A001, A002, A003, A004, A005, A006, A007...


In [16]:
traffic_summary = []

for client_id, data in datasets.items():

    timeline = data["timeline"]

    avg_vehicle = (
        timeline
        .groupby("timestamp")["car_id"]
        .nunique()
        .mean()
    )

    max_vehicle = (
        timeline
        .groupby("timestamp")["car_id"]
        .nunique()
        .max()
    )

    traffic_summary.append({

        "Client": client_id,

        "Avg Vehicles": round(avg_vehicle, 2),

        "Peak Vehicles": int(max_vehicle)

    })

traffic_df = pd.DataFrame(traffic_summary)

print(traffic_df)

  Client  Avg Vehicles  Peak Vehicles
0    DT1         32.84             39
1    DT2         32.96             41
2    DT3         33.01             43
